**Data Preprocessing**

1. Re-Orientation (RAS)
2. Brain Extraction
3. Intensity Normalisation (99th percentile not including background)
4. Resampling (1mm)
5. Resizing (128)

In [1]:
import os
import dotenv
dotenv.load_dotenv()
import sys
sys.path.append(os.getenv('PROJECT_ROOT'))
from src.data_preprocessor import MRIDataPreprocessor
import ants
import torch
import torchio as tio
import numpy as np

##### 1.0 MSLUB

In [2]:
# Define raw data path
mslub_t1_path = os.path.join(os.getenv('RAW_DATA_DIR'), 'MSLUB-T1')
mslub_t2_path = os.path.join(os.getenv('RAW_DATA_DIR'), 'MSLUB-T2')
mslub_gt_path = os.path.join(os.getenv('RAW_DATA_DIR'), 'MSLUB-GT')

# Define preprocessed data path
preprocessed_mslub_t1_path = os.path.join(os.getenv('PROJECT_ROOT'), 'data\MSLUB-T1')
preprocessed_mslub_t2_path = os.path.join(os.getenv('PROJECT_ROOT'), 'data\MSLUB-T2')
preprocessed_mslub_gt_path = os.path.join(os.getenv('PROJECT_ROOT'), 'data\MSLUB-GT')

In [3]:
# Initialize preprocessor
preprocessor = MRIDataPreprocessor()

In [4]:
# Preprocess T1 data
mslub_t1_file_count = 0

for t1_file in os.listdir(mslub_t1_path):
    file = os.path.join(mslub_t1_path, t1_file)
    ground_truth, noisy_img, mask = preprocessor.preprocess(file, 1, add_noise=False)
    save_path = os.path.join(preprocessed_mslub_t1_path, t1_file.replace('.nii.gz', '.npz'))
    preprocessor.save_to_npz(save_path, ground_truth, noisy_img, mask)
    mslub_t1_file_count += 1

if(mslub_t1_file_count == len(os.listdir(mslub_t1_path))):
    print('MSLUB T1 data preprocessing completed')
else:
    print('There are some files missing in MSLUB T1 data preprocessing')

MSLUB T1 data preprocessing completed


In [5]:
# Preprocess T2 data
mslub_t2_file_count = 0

for t2_file in os.listdir(mslub_t2_path):
    file = os.path.join(mslub_t2_path, t2_file)
    ground_truth, noisy_img, mask = preprocessor.preprocess(file, 2, add_noise=False)
    save_path = os.path.join(preprocessed_mslub_t2_path, t2_file.replace('.nii.gz', '.npz'))
    preprocessor.save_to_npz(save_path, ground_truth, noisy_img, mask)
    mslub_t2_file_count += 1

if(mslub_t2_file_count == len(os.listdir(mslub_t2_path))):
    print('MSLUB T2 data preprocessing completed')
else:
    print('There are some files missing in MSLUB T2 data preprocessing')

MSLUB T2 data preprocessing completed


In [6]:
# Preprocess GT data
mslub_gt_file_count = 0

for gt_file in os.listdir(mslub_gt_path):
    file = os.path.join(mslub_gt_path, gt_file)

    # Read as ants image
    ants_image = ants.image_read(file)

    # Reorient to RAS
    reoriented_image = ants.reorient_image2(ants_image, orientation='RAS')

    # Convert to torchio image
    torchio_image = tio.ScalarImage(tensor=torch.from_numpy(reoriented_image.numpy()).unsqueeze(0))

    # Resize to target shape
    resized_image = tio.Resize(target_shape=(128,128,128), image_interpolation='linear')(torchio_image)

    # Save as npz
    save_path = os.path.join(preprocessed_mslub_gt_path, gt_file.replace('.nii.gz', '.npz'))
    np.savez_compressed(save_path, data=resized_image.data)
    
    mslub_gt_file_count += 1

if(mslub_gt_file_count == len(os.listdir(mslub_gt_path))):
    print('MSLUB GT data preprocessing completed')
else:
    print('There are some files missing in MSLUB GT data preprocessing')

MSLUB GT data preprocessing completed
